In [1]:
import pandas as pd
print("pandas version:", pd.__version__)

pandas version: 3.0.3


In [2]:
df = pd.read_csv("data/results.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  str    
 1   home_team   49477 non-null  str    
 2   away_team   49477 non-null  str    
 3   home_score  49425 non-null  float64
 4   away_score  49425 non-null  float64
 5   tournament  49477 non-null  str    
 6   city        49477 non-null  str    
 7   country     49477 non-null  str    
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 3.1 MB


In [4]:
df["date"] = pd.to_datetime(df["date"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        49477 non-null  datetime64[us]
 1   home_team   49477 non-null  str           
 2   away_team   49477 non-null  str           
 3   home_score  49425 non-null  float64       
 4   away_score  49425 non-null  float64       
 5   tournament  49477 non-null  str           
 6   city        49477 non-null  str           
 7   country     49477 non-null  str           
 8   neutral     49477 non-null  bool          
dtypes: bool(1), datetime64[us](1), float64(2), str(5)
memory usage: 3.1 MB


In [5]:
df = df.sort_values("date").reset_index(drop=True)
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [6]:
print("Is the date column sorted ascending?", df["date"].is_monotonic_increasing)

Is the date column sorted ascending? True


In [7]:
print("Latest match date:", df["date"].max())
print(df[["date", "home_team", "away_team"]].tail())

Latest match date: 2026-06-27 00:00:00
            date home_team  away_team
49472 2026-06-27  Colombia   Portugal
49473 2026-06-27    Panama    England
49474 2026-06-27   Algeria    Austria
49475 2026-06-27    Jordan  Argentina
49476 2026-06-27   Croatia      Ghana


In [8]:
no_score = df[df["home_score"].isna()]
print("Matches with no score:", len(no_score))
print(no_score[["date", "home_team", "away_team", "tournament"]].head(60).to_string())

Matches with no score: 52
            date               home_team               away_team      tournament
49425 2026-06-17                Portugal                DR Congo  FIFA World Cup
49426 2026-06-17              Uzbekistan                Colombia  FIFA World Cup
49427 2026-06-17                 England                 Croatia  FIFA World Cup
49428 2026-06-17                   Ghana                  Panama  FIFA World Cup
49429 2026-06-18          Czech Republic            South Africa  FIFA World Cup
49430 2026-06-18                  Mexico             South Korea  FIFA World Cup
49431 2026-06-18             Switzerland  Bosnia and Herzegovina  FIFA World Cup
49432 2026-06-18                  Canada                   Qatar  FIFA World Cup
49433 2026-06-19                  Turkey                Paraguay  FIFA World Cup
49434 2026-06-19           United States               Australia  FIFA World Cup
49435 2026-06-19                Scotland                 Morocco  FIFA World Cup
49

In [9]:
from src.elo import compute_ratings

ratings = compute_ratings(df)
print("Number of teams rated:", len(ratings))

Number of teams rated: 336


In [10]:
top20 = sorted(ratings.items(), key=lambda x: x[1], reverse=True)[:20]

for rank, (team, rating) in enumerate(top20, start=1):
    print(f"{rank:2}. {team:<20} {rating:.0f}")

 1. Argentina            2054
 2. Spain                2033
 3. France               1996
 4. Portugal             1954
 5. Brazil               1946
 6. Germany              1946
 7. England              1944
 8. Colombia             1930
 9. Netherlands          1917
10. Japan                1912
11. Morocco              1908
12. Italy                1893
13. Croatia              1886
14. Mexico               1881
15. Belgium              1877
16. Ecuador              1871
17. Denmark              1853
18. Norway               1848
19. Turkey               1847
20. Austria              1847


In [11]:
# Only look at matches that were actually played (have scores).
played = df.dropna(subset=["home_score", "away_score"])

# A draw is when the two scores are equal.
draws = played[played["home_score"] == played["away_score"]]

draw_rate = len(draws) / len(played)
print(f"Played matches: {len(played)}")
print(f"Draws: {len(draws)}")
print(f"Draw rate: {draw_rate:.3f}  ({draw_rate*100:.1f}%)")

Played matches: 49425
Draws: 11241
Draw rate: 0.227  (22.7%)


In [12]:
from src.elo import predict_match
import numpy as np

# Use the final ratings we computed earlier (the `ratings` dict).
played = df.dropna(subset=["home_score", "away_score"])

draw_probs = []
for row in played.itertuples(index=False):
    r_home = ratings.get(row.home_team, 1500)
    r_away = ratings.get(row.away_team, 1500)
    probs = predict_match(r_home, r_away)
    draw_probs.append(probs["draw"])

print(f"Average predicted draw probability: {np.mean(draw_probs):.3f}")
print(f"Actual draw rate in the data:        0.227")

Average predicted draw probability: 0.226
Actual draw rate in the data:        0.227


In [13]:
import json
from datetime import datetime, timezone

# 21 June 2026 fixtures — names must match our dataset exactly.
fixtures = [
    ("Spain", "Saudi Arabia"),
    ("Belgium", "Iran"),
    ("Uruguay", "Cape Verde"),
    ("New Zealand", "Egypt"),
]

predictions = []
for home, away in fixtures:
    r_home = ratings.get(home, 1500)
    r_away = ratings.get(away, 1500)
    probs = predict_match(r_home, r_away)
    predictions.append({
        "home_team": home,
        "away_team": away,
        "home_rating": round(r_home),
        "away_rating": round(r_away),
        "prob_home_win": round(probs["win"], 3),
        "prob_draw": round(probs["draw"], 3),
        "prob_away_win": round(probs["loss"], 3),
    })

for p in predictions:
    print(f"{p['home_team']} vs {p['away_team']}: "
          f"H {p['prob_home_win']}  D {p['prob_draw']}  A {p['prob_away_win']}")

Spain vs Saudi Arabia: H 0.764  D 0.142  A 0.094
Belgium vs Iran: H 0.428  D 0.27  A 0.302
Uruguay vs Cape Verde: H 0.61  D 0.217  A 0.173
New Zealand vs Egypt: H 0.244  D 0.254  A 0.502


In [14]:
import json
from datetime import datetime, timezone

output = {
    "match_date": "2026-06-21",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "model": "elo-baseline-v1",
    "predictions": predictions,
}

with open("predictions/2026-06-21.json", "w") as f:
    json.dump(output, f, indent=2)

print("Saved predictions/2026-06-21.json")
print(json.dumps(output, indent=2))

Saved predictions/2026-06-21.json
{
  "match_date": "2026-06-21",
  "generated_at_utc": "2026-06-17T13:45:20.092717+00:00",
  "model": "elo-baseline-v1",
  "predictions": [
    {
      "home_team": "Spain",
      "away_team": "Saudi Arabia",
      "home_rating": 2033,
      "away_rating": 1669,
      "prob_home_win": 0.764,
      "prob_draw": 0.142,
      "prob_away_win": 0.094
    },
    {
      "home_team": "Belgium",
      "away_team": "Iran",
      "home_rating": 1877,
      "away_rating": 1817,
      "prob_home_win": 0.428,
      "prob_draw": 0.27,
      "prob_away_win": 0.302
    },
    {
      "home_team": "Uruguay",
      "away_team": "Cape Verde",
      "home_rating": 1845,
      "away_rating": 1626,
      "prob_home_win": 0.61,
      "prob_draw": 0.217,
      "prob_away_win": 0.173
    },
    {
      "home_team": "New Zealand",
      "away_team": "Egypt",
      "home_rating": 1632,
      "away_rating": 1757,
      "prob_home_win": 0.244,
      "prob_draw": 0.254,
      "prob_